# TP DES & AES — Partie 7 : AES vs DES

## Introduction

Le **DES** (Data Encryption Standard) a été publié en 1977 par le NBS (aujourd'hui NIST), à partir du chiffrement Lucifer d'IBM modifié avec la NSA. Pendant plus de 20 ans, il a été le standard du chiffrement symétrique (banques, cartes à puce, etc.). À la fin des années 1990 sa clé de 56 bits devient trop courte, et le NIST lance en 1997 un concours ouvert pour le remplacer. En 2001, l'algorithme **Rijndael** (Joan Daemen et Vincent Rijmen) est choisi et standardisé sous le nom d'**AES** (FIPS 197). On compare ici les deux algorithmes et on explique en quoi AES améliore la sécurité.

## 1. Taille de clé et taille de bloc

| | DES | AES |
|---|---|---|
| Taille de bloc | 64 bits | 128 bits |
| Taille de clé | 56 bits effectifs (64 avec parité) | 128, 192 ou 256 bits |
| Nombre de tours | 16 | 10 / 12 / 14 |
| Nombre de clés | $2^{56} \approx 7{,}2\cdot10^{16}$ | $2^{128} \approx 3{,}4\cdot10^{38}$ (min) |

**Clé.** La clé de 56 bits est le principal point faible de DES. Elle est le résultat d'un compromis (IBM proposait plus, la NSA a imposé 56 bits). $2^{56}$ clés étaient déjà considérées comme « limite » en 1977, et l'augmentation de la puissance de calcul a fini par rendre la force brute réaliste. AES utilise au minimum 128 bits, soit $2^{72}$ fois plus de clés que DES.

**Bloc.** La taille de bloc a aussi un impact sur la sécurité, souvent moins visible. Avec un bloc de $n$ bits, d'après le paradoxe des anniversaires, on commence à avoir des collisions de blocs chiffrés après environ $2^{n/2}$ blocs. Pour DES ($n=64$), cela fait $2^{32}$ blocs = 32 Go : c'est atteint en pratique (attaque **Sweet32** en 2016 contre 3DES et Blowfish dans TLS/OpenVPN, qui récupère des cookies). Pour AES ($n=128$), il faut $2^{64}$ blocs ≈ 256 Eio, ce qui n'est pas atteignable. Un bloc plus grand permet donc de chiffrer beaucoup plus de données avec la même clé, en mode CBC ou CTR.

## 2. Structure du chiffrement et conception des tours

### DES : réseau de Feistel

À chaque tour, seule **une moitié** du bloc (32 bits) est transformée par la fonction $f$, puis XORée à l'autre moitié : $L_{i+1} = R_i$, $R_{i+1} = L_i \oplus f(R_i, K_i)$. La fonction $f$ comprend l'expansion E (32→48), le XOR avec la sous-clé, les 8 S-box 6→4 bits et la permutation P. L'avantage est que $f$ n'a pas besoin d'être inversible et que le déchiffrement utilise le même circuit. L'inconvénient est que la diffusion est plus lente (il faut au moins 2 tours pour que tous les bits soient touchés), d'où un grand nombre de tours (16).

Les S-box de DES ont été conçues en secret (critères publiés seulement en 1994) ce qui a longtemps alimenté des soupçons de porte dérobée. On sait aujourd'hui qu'elles avaient été renforcées contre la cryptanalyse différentielle, connue d'IBM et de la NSA mais pas du public.

### AES : réseau de substitution–permutation (SPN)

AES n'est **pas** un Feistel : à chaque tour, **tout le bloc** de 128 bits (vu comme une matrice 4×4 d'octets, l'« état ») est transformé. Un tour comprend 4 étapes :

1. **SubBytes** : chaque octet passe dans une S-box 8→8 bits. Cette S-box n'est pas arbitraire : elle est définie par l'inversion dans le corps fini $GF(2^8)$ suivie d'une transformation affine. Elle a des propriétés quasi optimales contre les cryptanalyses linéaire et différentielle (probabilité différentielle maximale $2^{-6}$, biais linéaire maximal $2^{-4}$).
2. **ShiftRows** : décalage circulaire des lignes de 0, 1, 2 et 3 octets → les octets d'une colonne sont envoyés dans des colonnes différentes.
3. **MixColumns** : chaque colonne est multipliée par une matrice fixe dans $GF(2^8)$. C'est un code MDS : un octet modifié en entrée modifie les 4 octets de la colonne en sortie (nombre de branches = 5).
4. **AddRoundKey** : XOR avec la sous-clé de 128 bits.

Le dernier tour n'a pas de MixColumns, et une AddRoundKey initiale est faite avant le premier tour. Le déchiffrement utilise les transformations inverses (InvSubBytes, InvShiftRows, InvMixColumns).

Grâce à ShiftRows + MixColumns, **après 2 tours chaque bit de l'état dépend de tous les bits d'entrée** (diffusion complète). C'est la « wide trail strategy » des auteurs : on peut **prouver** une borne sur le nombre minimum de S-box actives sur 4 tours (au moins 25), donc une borne sur la probabilité de toute caractéristique différentielle ou linéaire ($\le 2^{-150}$), ce qui rend ces attaques inapplicables. DES n'avait pas ce type de garantie : sa résistance a été montrée de manière empirique.

### Cadencement de clé

- DES : sélection de bits (PC-1, décalages, PC-2), c'est uniquement une permutation des bits de la clé, sans aucune non-linéarité. Cela entraîne des faiblesses : **4 clés faibles** (pour lesquelles toutes les sous-clés sont égales, et chiffrer deux fois redonne le clair) et 6 paires de clés semi-faibles. On a aussi la propriété de **complémentation** : $E_{\bar K}(\bar P) = \overline{E_K(P)}$, qui divise par 2 l'effort d'une recherche exhaustive.
- AES : le key schedule utilise la S-box (non linéaire), des rotations et des constantes de tour (Rcon). Il n'y a ni clé faible connue ni propriété de complémentation.

### Implémentation

DES a été conçu pour le matériel des années 70 : ses permutations de bits sont gratuites en câblage mais coûteuses en logiciel (on l'a vu dans la partie 4, il a fallu des tables pour accélérer). AES fonctionne sur des octets et des mots de 32 bits, il est efficace en logiciel (tables T de 32 bits) et en matériel. Les processeurs modernes ont même des instructions dédiées (**AES-NI** chez Intel/AMD, extensions ARMv8), qui permettent des débits de plusieurs Go/s et suppriment les attaques par canal auxiliaire liées au cache sur les tables. AES est en pratique beaucoup plus rapide que 3DES.

## 3. Pourquoi AES résiste mieux à la force brute

Le nombre de clés suffit pour comparer. Estimons le temps de recherche exhaustive (en moyenne il faut tester la moitié des clés) :

In [1]:
rates = {"PC (1e9 clés/s)": 1e9,
         "Deep Crack 1998 (9e10 clés/s)": 9e10,
         "Cluster FPGA/GPU moderne (1e12 clés/s)": 1e12,
         "Hypothétique (1e18 clés/s)": 1e18}
keys = {"DES (56 bits)": 2**56, "AES-128": 2**128, "AES-256": 2**256}
YEAR = 3600 * 24 * 365.25

def fmt(s):
    if s < 3600: return f"{s:.3g} s"
    if s < 86400 * 2: return f"{s/3600:.3g} h"
    if s < YEAR: return f"{s/86400:.3g} jours"
    return f"{s/YEAR:.3g} ans"

print(f"{'':42s}" + "".join(f"{k:>16s}" for k in keys))
for r, v in rates.items():
    print(f"{r:42s}" + "".join(f"{fmt(n / 2 / v):>16s}" for n in keys.values()))
print(f"\n(âge de l'univers ≈ 1.38e10 ans)")

                                             DES (56 bits)         AES-128         AES-256
PC (1e9 clés/s)                                   1.14 ans    5.39e+21 ans    1.83e+60 ans
Deep Crack 1998 (9e10 clés/s)                   4.63 jours    5.99e+19 ans    2.04e+58 ans
Cluster FPGA/GPU moderne (1e12 clés/s)                10 h    5.39e+18 ans    1.83e+57 ans
Hypothétique (1e18 clés/s)                         0.036 s    5.39e+12 ans    1.83e+51 ans

(âge de l'univers ≈ 1.38e10 ans)


Historiquement, DES a été cassé par force brute en **1997** (projet DESCHALL, plusieurs mois de calcul distribué), puis en **1998** par la machine **Deep Crack** de l'EFF (~250 000 $) en 56 heures, et en **1999** en 22 heures (Deep Crack + distributed.net). Aujourd'hui, une machine FPGA comme COPACOBANA ou des services en ligne cassent une clé DES en moins d'une journée.

Pour AES-128, même avec un milliard de milliards de clés par seconde, il faudrait $\approx 5\cdot 10^{12}$ années, bien plus que l'âge de l'univers. Ajouter 1 bit de clé double le travail : passer de 56 à 128 bits multiplie l'effort par $2^{72} \approx 4,7 \cdot 10^{21}$. Même l'algorithme quantique de Grover, qui ne donne qu'une accélération quadratique ($2^{n/2}$), laisse AES-256 à un niveau de sécurité de 128 bits, ce qui est la raison pour laquelle AES-256 est recommandé pour la sécurité à long terme.

## 4. Différences en cryptanalyse

**DES :**
- **Cryptanalyse différentielle** (Biham & Shamir, 1990) : nécessite $2^{47}$ clairs choisis. C'est mieux que la force brute en nombre d'opérations, mais peu réaliste à cause de la quantité de clairs choisis. Les S-box de DES avaient justement été optimisées contre cette attaque.
- **Cryptanalyse linéaire** (Matsui, 1993) : $2^{43}$ clairs connus, réalisée expérimentalement par Matsui en 1994. C'est la meilleure attaque analytique connue sur DES complet.
- Propriété de complémentation et clés faibles (voir plus haut).
- Contre-mesure : **3DES** (Triple DES, EDE avec 2 ou 3 clés). La sécurité effective est de 112 bits (3 clés) à cause de l'attaque « meet-in-the-middle », qui explique aussi pourquoi le double DES n'apporte presque rien ($2^{57}$ au lieu de $2^{112}$). Mais 3DES est 3 fois plus lent et garde le bloc de 64 bits (Sweet32). Le NIST l'a déclaré obsolète, son utilisation est interdite depuis fin 2023.

**AES :**
- La conception est publique et justifiée mathématiquement (wide trail), et AES résiste à la cryptanalyse linéaire et différentielle avec une grande marge.
- La meilleure attaque sur AES complet est l'attaque **biclique** (Bogdanov, Khovratovich, Rechberger, 2011) : $2^{126,1}$ opérations pour AES-128, soit un gain d'un facteur ~4 par rapport à la force brute, sans aucun intérêt pratique.
- Des attaques existent sur des versions **réduites** (par ex. 7 tours sur 10 pour AES-128) ou dans des modèles particuliers (attaques à clés liées sur AES-192/256, sur le key schedule), mais aucune ne menace AES utilisé normalement.
- Les attaques pratiques contre AES visent l'**implémentation** (canaux auxiliaires : timing de cache sur les tables, consommation, injection de fautes) et non l'algorithme. Elles sont limitées par les instructions AES-NI et des implémentations à temps constant.

Après plus de 20 ans d'analyse intensive par la communauté, AES n'a perdu que ~2 bits de sécurité, alors que DES a été cassé par force brute 20 ans après sa publication.

## 5. Conclusion : pourquoi AES a remplacé DES

1. **Clé trop courte** : 56 bits est cassable par force brute depuis 1998 avec du matériel accessible. AES offre 128 à 256 bits, hors de portée même à long terme.
2. **Bloc trop petit** : 64 bits limite la quantité de données chiffrables avec une même clé (collisions, Sweet32). AES utilise 128 bits.
3. **Une conception plus solide et transparente** : AES a été choisi lors d'un concours public et international (15 candidats, 3 ans d'analyse), avec une justification mathématique de sa résistance aux attaques linéaires et différentielles, alors que les critères de conception de DES sont restés secrets pendant 17 ans.
4. **Performances** : 3DES, la solution provisoire, est lent (48 tours) et peu adapté au logiciel. AES est rapide en logiciel comme en matériel, et accéléré nativement par les processeurs actuels.
5. **Flexibilité** : trois tailles de clés pour s'adapter au niveau de sécurité voulu, une marge face aux attaques quantiques avec AES-256.

DES reste un algorithme très important historiquement et pédagogiquement (réseau de Feistel, S-box, confusion/diffusion, origine des cryptanalyses linéaire et différentielle), mais il ne doit plus être utilisé. AES est aujourd'hui le standard de chiffrement symétrique utilisé partout (TLS, Wi-Fi WPA2/3, chiffrement de disques, VPN…).

### Références
- W. Stallings, *Cryptography and Network Security*, 8e édition, Pearson, 2023 (chapitres 4 et 6)
- NIST FIPS 46-3 (DES) et FIPS 197 (AES)
- J. Daemen, V. Rijmen, *The Design of Rijndael*, Springer, 2002
- K. Bhargavan, G. Leurent, *On the Practical (In-)Security of 64-bit Block Ciphers* (Sweet32), CCS 2016